# Topic 8 — Logistic Regression
### Theory → tiny example → from-scratch implementation → sklearn → decision boundary plot.

Despite the name, logistic regression is a **classification** algorithm, not regression.
It predicts the *probability* that a sample belongs to class 1, then applies a threshold.

```text
z = wx + b                (same linear combination as linear regression)
sigmoid(z)  -> probability (squashes z into range (0, 1))
threshold   -> class       (e.g. probability >= 0.5 -> class 1, else class 0)
```

This is one of the most important algorithms for your cyberbullying paper — it's a standard
baseline for binary text classification (bullying vs not-bullying).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

rng = np.random.default_rng(0)

## 1. The sigmoid function

Sigmoid squashes any real number into the range (0, 1), so it can be read as a probability.

```text
sigmoid(z) = 1 / (1 + e^(-z))
```

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.linspace(-10, 10, 200)
plt.figure(figsize=(5, 4))
plt.plot(z_values, sigmoid(z_values))
plt.axhline(0.5, color="gray", linestyle="--", lw=0.8)
plt.axvline(0, color="gray", linestyle="--", lw=0.8)
plt.title("Sigmoid function")
plt.xlabel("z"); plt.ylabel("sigmoid(z)")
plt.show()
# Large positive z -> probability near 1. Large negative z -> probability near 0.
# z = 0 -> probability exactly 0.5 (the default decision boundary).

## 2. Binary classification, probability, threshold, decision boundary

- **Decision boundary**: the line (or surface) where the model is exactly 50/50 undecided.
- **Threshold**: the probability cutoff used to turn a probability into a hard class label (commonly 0.5,
  but can be tuned — see Topic 9).

In [ ]:
# 2D toy dataset, 2 classes
X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.5, random_state=42
)

plt.figure(figsize=(5, 4))
plt.scatter(X[y==0, 0], X[y==0, 1], label="class 0", alpha=0.7)
plt.scatter(X[y==1, 0], X[y==1, 1], label="class 1", alpha=0.7)
plt.legend()
plt.title("2D binary classification data")
plt.show()

## 3. Log loss (binary cross-entropy)

MSE (used for regression) doesn't work well for classification. Instead we use **log loss**:

```text
loss = -[ y*log(p) + (1-y)*log(1-p) ]
```

It heavily penalizes *confident and wrong* predictions (e.g. predicting p=0.99 for class 0 when the
true label is 1 gives a huge loss), which is exactly the behavior you want from a classifier.

In [ ]:
def log_loss(y_true, p_pred, eps=1e-12):
    p_pred = np.clip(p_pred, eps, 1 - eps)   # avoid log(0)
    return -np.mean(y_true * np.log(p_pred) + (1 - y_true) * np.log(1 - p_pred))

# Illustrate: being confidently WRONG costs a lot more than being unsure
print("loss if p=0.5 for true label 1 (unsure):        ", log_loss(np.array([1]), np.array([0.5])))
print("loss if p=0.9 for true label 1 (confident+right):", log_loss(np.array([1]), np.array([0.9])))
print("loss if p=0.1 for true label 1 (confident+wrong):", log_loss(np.array([1]), np.array([0.1])))

## 4. From-scratch implementation with gradient descent

In [ ]:
def train_logistic_regression(X, y, lr=0.1, epochs=2000):
    n, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    loss_history = []

    for epoch in range(epochs):
        z = X @ w + b
        p = sigmoid(z)
        loss = log_loss(y, p)
        loss_history.append(loss)

        # gradients of log loss w.r.t. w and b (derived via calculus, same chain-rule idea as Topic 4)
        dw = (1/n) * X.T @ (p - y)
        db = (1/n) * np.sum(p - y)

        w -= lr * dw
        b -= lr * db

    return w, b, loss_history

# Standardize features first -- helps gradient descent converge
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
w, b, loss_history = train_logistic_regression(X_norm, y, lr=0.5, epochs=1000)

print("learned w:", w, " learned b:", b)

plt.figure(figsize=(5, 4))
plt.plot(loss_history)
plt.xlabel("epoch"); plt.ylabel("log loss")
plt.title("Log loss decreasing during training")
plt.show()

## 5. sklearn implementation

In [ ]:
clf = LogisticRegression()
clf.fit(X, y)

print("sklearn learned coef_:", clf.coef_)
print("sklearn learned intercept_:", clf.intercept_)

# predict_proba gives the actual probability, not just the hard class
sample = X[:5]
print("predicted probabilities (class0, class1):\n", clf.predict_proba(sample))
print("predicted classes:", clf.predict(sample))
print("true labels:       ", y[:5])

## 6. Plot the decision boundary

Visualizing where the model switches from predicting class 0 to class 1.

In [ ]:
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

grid_probs = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, grid_probs, levels=25, cmap="RdBu_r", alpha=0.6)
plt.colorbar(label="P(class 1)")
plt.contour(xx, yy, grid_probs, levels=[0.5], colors="black", linewidths=2)   # the decision boundary itself
plt.scatter(X[y==0, 0], X[y==0, 1], edgecolor="k", label="class 0")
plt.scatter(X[y==1, 0], X[y==1, 1], edgecolor="k", label="class 1")
plt.legend()
plt.title("Decision boundary (black line = 50/50 threshold)")
plt.show()

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Regenerate the dataset with class_sep=0.5 (much more overlap) and re-plot the decision boundary.
#    Notice the model still draws a straight line -- logistic regression can only learn LINEAR boundaries.
# 2. In the from-scratch section, change lr to 0.01 and 2.0 -- compare how fast loss decreases.
# 3. Using clf.predict_proba, find how many samples have a predicted probability between 0.4 and 0.6
#    (the "uncertain" zone near the boundary).
# 4. Look up why we DON'T use MSE as the loss for logistic regression (hint: it's non-convex here --
#    gradient descent can get stuck). One sentence is enough.

---
### Next up: **Topic 9 — Classification Evaluation** (confusion matrix, precision/recall/F1 — essential for imbalanced data like cyberbullying detection).

Say "next" when you're ready.